# Classic SPL v9.4.0 — Waterfall & Univariate Validation (cat excluded)

Compares the new release (v9.4.0, branch `B-2937390`) against the previously
shipped release (v9.3.1), per-policy. Self-contained: run top to bottom.

**Two values depend on infrastructure and are marked `# CONFIRM` in the config
cell.** If either is wrong the notebook stops at the cell that uses it, with an
error message that hands you the fix:
1. the release path tokens (`NB_SPL_v9.4.0` / `v9.3.1`)
2. the git branch string (`feature/B-2937390`)

Only the fan-out check hard-stops. A waterfall that doesn't close prints a
warning with the residual instead of halting — the residual is the size of any
component you're not accounting for.

> Classic baseline caveat: premium differs per policy between releases, so the
> full-population waterfall mixes fit movement with pipeline drift. Charts below
> run on the **stable subset** (premium unchanged) for that reason.

In [ ]:
# --- env: MUST run before the classic_spl_ltv import below. --------------
# If you re-run after importing, restart the kernel first.
%env ENV_FOR_DYNACONF=prod
%env DYNACONF_GIT_BRANCH=feature/B-2937390
%env DYNACONF_GIT_CHECKOUT=feature/B-2937390

In [ ]:
# --- imports + config -----------------------------------------------------
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import cm

import ltv_helpers.non_spark_helpers as nsh
from classic_spl_ltv.config.paths import paths as p

pd.options.display.max_columns = 500
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- CONFIRM these two against the printed path in the next cell ----------
NEW_RELEASE = "release/NB_SPL_v9.4.0"   # CONFIRM: matches token in new_path
OLD_RELEASE = "release/NB_SPL_v9.3.1"   # CONFIRM: previously shipped release
# --------------------------------------------------------------------------

ID_COL      = "adw_pol_id"
MERGE_KEYS  = ["adw_pol_id", "release_day"]   # id alone fans out; both needed
DRIFT_COL   = "drv_full_premium_amt"          # premium; drives the classic caveat
OLD         = "_old"                          # suffix put on baseline columns
EXP_PREFIX  = re.compile(r"^e\d{3}scl_")       # classic=e005scl_, discovered at runtime

LINE_NAMES = {16: "Specialty Auto", 32: "Mfg Home", 71: "Renters",
              72: "Landlord", 78: "Condo", 88: "PUP", 90: "Boat"}

In [ ]:
# --- resolve both release paths -------------------------------------------
new_path = p.score_internal_results
print("new_path:", new_path)


def swap_release(path, new, old):
    if new not in path:
        raise ValueError(
            f"NEW_RELEASE {new!r} not found in path:\n  {path}\n"
            f"Edit NEW_RELEASE / OLD_RELEASE to match the release token above."
        )
    return path.replace(new, old)


old_path = swap_release(new_path, NEW_RELEASE, OLD_RELEASE)
print("old_path:", old_path)

In [ ]:
# --- load both releases ---------------------------------------------------
# Reads all columns so a renamed field shows up as a visible diff rather than a
# KeyError later. If memory is tight, pass columns=[...] to both reads.
df_new = nsh.read_parquet_s3_to_pandas(new_path)
df_old = nsh.read_parquet_s3_to_pandas(old_path)
print("new:", df_new.shape, " old:", df_old.shape)

In [ ]:
# --- strip the e0NNscl_ expense prefix ------------------------------------
def strip_exp_prefix(df):
    found = sorted({m.group(0) for c in df.columns if (m := EXP_PREFIX.match(c))})
    out = df.rename(columns=lambda c: EXP_PREFIX.sub("", c))
    dupes = out.columns[out.columns.duplicated()].tolist()
    if dupes:
        raise ValueError(f"prefix strip collided on: {dupes}")
    return out, found


df_new, pre_new = strip_exp_prefix(df_new)
df_old, pre_old = strip_exp_prefix(df_old)
print("expense prefix  new:", pre_new, " old:", pre_old)
# If these differ between releases that is expected (it happened between renters
# v1.2 and v1.4). Just confirm the stripped names line up before trusting diffs.

In [ ]:
# --- merge on [adw_pol_id, release_day]; guard fan-out, report drift -------
def merge_releases(df_new, df_old, keys, drift_col=DRIFT_COL):
    for name, d in (("new", df_new), ("old", df_old)):
        if d.duplicated(keys).any():
            raise ValueError(f"{name} release is not unique on {keys}")

    old = df_old.rename(columns={c: c + OLD for c in df_old.columns if c not in keys})
    out = df_new.merge(old, how="inner", on=keys)

    if len(out) > max(len(df_new), len(df_old)):
        raise ValueError(
            f"fan-out: new={len(df_new):,} old={len(df_old):,} -> {len(out):,} on {keys}"
        )
    print(f"merged on {keys}: new={len(df_new):,} old={len(df_old):,} "
          f"matched={len(out):,} ({len(out)/len(df_new):.1%} of new)")

    if drift_col in out and drift_col + OLD in out:
        d = (out[drift_col] - out[drift_col + OLD]).abs()
        print(f"{drift_col} drift: median|d|={d.median():,.2f} max={d.max():,.2f} "
              f"nulls={d.isna().sum():,} pct_exact={np.isclose(d.fillna(1), 0).mean():.1%}")
    return out


combined = merge_releases(df_new, df_old, MERGE_KEYS)
if "release_day" in combined:
    combined["release_mth_yr"] = combined["release_day"].astype(str).str[0:7]

In [ ]:
# --- diffs: old - new, for whatever waterfall columns are present ---------
WF_COLS = [
    "lifetime_premium", "lifetime_loss", "balance_amt", "cat_loss_amt",
    "commission_exp_new", "commission_exp_renew", "investment_income", "tax",
    "ltv", "aac_mkt", "cost_of_capital", "cost_of_capital_20pct",
    "acquisition_exp", "claims_exp", "lifetime_exp", "ple",
]


def add_diffs(df, cols):
    made, missing = [], []
    for c in cols:
        if c in df and c + OLD in df:
            df[f"{c}_diff"] = df[c + OLD] - df[c]
            made.append(c)
        else:
            missing.append(c)
    print("diffs computed for:", made)
    if missing:
        print("NO diff (column absent this release):", missing)
    return df


combined = add_diffs(combined, WF_COLS)

In [ ]:
# --- stable subset: policies whose premium didn't move between releases ---
# Use this for the classic waterfall so fit movement isn't buried under
# pipeline-state drift. Report coverage on any slide built from it.
def stable_subset(df, drift_col=DRIFT_COL, atol=0.01):
    same = np.isclose(df[drift_col], df[drift_col + OLD], atol=atol, equal_nan=True)
    print(f"stable subset: {same.sum():,} / {len(df):,} ({same.mean():.1%})")
    return df.loc[same].copy()


stable = stable_subset(combined)

## II. Univariate compares

Bars = policy count per group, lines = mean of each metric for the two
releases. Runs on the full merged frame. Metrics or group columns that aren't
present are skipped with a note, so this cell always completes.

In [ ]:
def plot_counts_with_two_lines(df, group_col, count_col, value_cols,
                               line_labels=None, title=None, xlabel=None,
                               ylabel_left="Count of Policies", ylabel_right=None,
                               rotate_xticks=False):
    grouped = (
        df.groupby(group_col, dropna=False)
        .agg(count=(count_col, "count"),
             v1=(value_cols[0], "mean"),
             v2=(value_cols[1], "mean"))
        .reset_index()
    )
    x = range(len(grouped))
    figw = max(8, 0.28 * len(grouped))
    fig, ax1 = plt.subplots(figsize=(figw, 5))
    ax1.bar(x, grouped["count"], width=0.6, label="Count")
    ax1.set_xlabel(xlabel or group_col)
    ax1.set_ylabel(ylabel_left)
    ax1.set_xticks(list(x))
    ax1.set_xticklabels(grouped[group_col], rotation=90 if rotate_xticks else 0)
    ax2 = ax1.twinx()
    for i, col in enumerate(["v1", "v2"]):
        ax2.plot(x, grouped[col], color=["blue", "orange"][i], marker="o",
                 label=line_labels[i] if line_labels else value_cols[i])
    ax2.set_ylim(bottom=0)
    ax2.set_ylabel(ylabel_right or "mean")
    ax1.legend(loc="upper left")
    ax2.legend(loc="upper right")
    plt.title(title or group_col)
    plt.tight_layout()
    plt.show()

In [ ]:
group_dict = {
    "drv_chnl_of_bnd": "Channel of Bind",
    "ply_pt_state_cd": "State",
    "release_mth_yr":  "Release Month",
    "drv_line":        "Line",
}
metrics = [("ltv", "Average ($)"), ("ple", "PLE (years)"), ("aac_mkt", "Average AAC ($)")]

for metric, ylab in metrics:
    if metric not in combined or metric + OLD not in combined:
        print(f"skip metric {metric!r}: not in both releases")
        continue
    for gcol, gname in group_dict.items():
        if gcol not in combined:
            print(f"skip group {gcol!r}: absent")
            continue
        plot_counts_with_two_lines(
            df=combined, group_col=gcol, count_col=ID_COL,
            value_cols=[metric, metric + OLD],
            line_labels=[f"v9.4.0 {metric}", f"v9.3.1 {metric}"],
            title=f"{metric.upper()}: v9.3.1 vs v9.4.0  by {gname}",
            xlabel=gname, ylabel_right=ylab,
            rotate_xticks=gcol in ("ply_pt_state_cd", "release_mth_yr"),
        )

## III. Waterfall plots

Anchor bar = old-release metric, final bar = new-release metric, middle bars =
each component's `old - new` contribution with its sign. If the bars don't land
on the final value a component is missing or mis-signed — that prints as a
warning with the residual; the chart still draws.

In [ ]:
# signs apply to the *_diff columns (diff = old - new)
LTV_SIGNS = {
    "lifetime_premium": -1, "lifetime_loss": 1, "balance_amt": 1, "cat_loss_amt": 1,
    "claims_exp": 1, "lifetime_exp": 1, "commission_exp_renew": 1,
    "cost_of_capital": 1, "tax": 1, "investment_income": -1,
}
AAC_SIGNS = {
    "lifetime_premium": -1, "lifetime_loss": 1, "balance_amt": 1, "cat_loss_amt": 1,
    "claims_exp": 1, "lifetime_exp": 1, "commission_exp_renew": 1,
    "cost_of_capital_20pct": 1, "tax": 1, "investment_income": -1,
    "commission_exp_new": 1, "acquisition_exp": 1,
}
LABELS = {
    "lifetime_premium": "Lifetime Premium", "lifetime_loss": "Lifetime Loss",
    "balance_amt": "Balance", "cat_loss_amt": "Cat Loss",
    "claims_exp": "Claims Expense", "lifetime_exp": "Lifetime Expense",
    "commission_exp_renew": "Commission Renew", "commission_exp_new": "Commission New",
    "acquisition_exp": "Acquisition Expense", "cost_of_capital": "Cost of Capital",
    "cost_of_capital_20pct": "Cost of Capital (20%)", "tax": "Tax",
    "investment_income": "Investment Income",
}


def build_components(metric, signs, row_index):
    """Anchor first, final last; only include *_diff columns actually present."""
    comps = {f"{metric}{OLD}": (f"v9.3.1 {metric.upper()}", 1)}
    for col, sign in signs.items():
        dcol = f"{col}_diff"
        if dcol in row_index:
            comps[dcol] = (f"{LABELS.get(col, col)} Diff", sign)
    comps[metric] = (f"v9.4.0 {metric.upper()}", 0)
    return comps


def make_waterfall(row, components, title, atol=0.5, ax=None):
    order = list(components)
    for c in (order[0], order[-1]):
        if c not in row.index:
            print(f"skip {title!r}: anchor/final column {c!r} missing")
            return None
    labels = [components[c][0] for c in order]
    signs  = np.array([components[c][1] for c in order])
    values = np.array([float(row[c]) for c in order])

    heights = [values[0]] + list(values[1:-1] * signs[1:-1]) + [values[-1]]
    implied = heights[0] + sum(heights[1:-1])
    resid   = implied - values[-1]
    if abs(resid) > atol:
        print(f"WATERFALL DOES NOT CLOSE for {title!r}: anchor+diffs={implied:,.2f} "
              f"vs final={values[-1]:,.2f} (residual {resid:,.2f}). "
              f"A component is missing / double-counted / wrong-signed.")

    bottoms, cum = [0], values[0]
    for h in heights[1:-1]:
        bottoms.append(cum); cum += h
    bottoms.append(0)

    if ax is None:
        _, ax = plt.subplots(figsize=(12, 6))
    ax.bar(labels, heights, bottom=bottoms, color=cm.tab20(np.arange(len(heights))))
    for i, (b, h) in enumerate(zip(bottoms, heights)):
        ax.text(i, b + h, f"{h:,.0f}", ha="center", va="bottom", fontsize=9)
    ax.axhline(implied, color="red", linestyle="--", label="Final $")
    ax.set_ylabel("Value")
    ax.set_title(f"{title}   (residual {resid:+,.2f})")
    ax.legend()
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.tight_layout(); plt.show()
    return resid

In [ ]:
# quick look at what the waterfall has to work with
diff_cols = sorted(c for c in stable.columns if c.endswith("_diff"))
print("available *_diff columns:")
for c in diff_cols:
    print("  ", c)

In [ ]:
# --- overall waterfalls (on the stable subset) ---------------------------
frame = stable   # switch to `combined` to see the drift-contaminated version

ltv_comps = build_components("ltv", LTV_SIGNS, frame.columns)
aac_comps = build_components("aac_mkt", AAC_SIGNS, frame.columns)

overall = frame[[c for c in set(ltv_comps) | set(aac_comps) if c in frame]].mean()

make_waterfall(overall, ltv_comps, "Overall LTV -- classic v9.3.1 -> v9.4.0")
make_waterfall(overall, aac_comps, "Overall AAC -- classic v9.3.1 -> v9.4.0")

In [ ]:
# --- by line (classic fits differ per line) ------------------------------
by_line = (
    frame.groupby("drv_line")[[c for c in set(ltv_comps) | set(aac_comps) if c in frame]]
    .mean()
)
for line, row in by_line.iterrows():
    name = LINE_NAMES.get(line, line)
    make_waterfall(row, ltv_comps, f"LTV v9.3.1 -> v9.4.0 -- {name} ({line})")